# 🎓 Logistic Regression — Student Admission Prediction
---
**Dataset:** `ex2data1.txt` — 100 students, 2 exam scores, 1 admission outcome

| Column | Description |
|--------|-------------|
| `Exam1` | Score on first exam |
| `Exam2` | Score on second exam |
| `Admitted` | 1 = Admitted, 0 = Not Admitted |

**Tasks:**
1. Data Exploration & Visualization
2. Logistic Regression with scikit-learn
3. Making Predictions
4. Model Evaluation

---
## ⚙️ Setup — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

---
## 📂 Upload the Dataset

In [ ]:
from google.colab import files
print('📂 Please upload your ex2data1.txt file:')
uploaded = files.upload()

---
## 🌟 Part 1 — Data Exploration

In [ ]:
# ── Load the dataset ──────────────────────────────────────────
df = pd.read_csv('ex2data1.txt', header=None, names=['Exam1', 'Exam2', 'Admitted'])

print('Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

In [ ]:
# ── Basic statistics ──────────────────────────────────────────
print('\n📊 Descriptive Statistics:')
display(df.describe().round(2))

print('\n🎯 Target Distribution:')
counts = df['Admitted'].value_counts()
print(f'  Admitted (1)     : {counts[1]} students ({counts[1]/len(df)*100:.1f}%)')
print(f'  Not Admitted (0) : {counts[0]} students ({counts[0]/len(df)*100:.1f}%)')

print('\n🔍 Missing Values:', df.isnull().sum().sum())

In [ ]:
# ── Scatter Plot: Admitted vs Not Admitted ────────────────────
admitted     = df[df['Admitted'] == 1]
not_admitted = df[df['Admitted'] == 0]

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

ax.scatter(admitted['Exam1'],     admitted['Exam2'],
           c='#10b981', s=80, marker='o', edgecolors='white',
           linewidths=0.5, label='Admitted (1)', zorder=3)
ax.scatter(not_admitted['Exam1'], not_admitted['Exam2'],
           c='#ef4444', s=80, marker='x', linewidths=1.8,
           label='Not Admitted (0)', zorder=3)

ax.set_xlabel('Exam 1 Score', color='white', fontsize=12)
ax.set_ylabel('Exam 2 Score', color='white', fontsize=12)
ax.set_title('Student Admissions\nExam 1 Score vs Exam 2 Score',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#0f172a', labelcolor='white', fontsize=11)
for sp in ax.spines.values(): sp.set_edgecolor('#334155')
ax.grid(True, color='#334155', linewidth=0.5, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print('\n📌 Observation: Students with higher scores on both exams tend to be admitted.')
print('   A clear diagonal boundary separates most admitted from non-admitted students.')

### 📌 Data Exploration — Findings

The dataset contains **100 student records** with no missing values. Each student has two exam scores and a binary admission outcome. The scatter plot reveals a clear pattern: students who scored high on **both** exams were generally admitted, while students with low scores on either exam were not. This visual separation suggests that a linear decision boundary (as produced by Logistic Regression) should perform well on this data.

---
## 🌟 Part 2 — Logistic Regression with scikit-learn

In [ ]:
# ── Prepare features and target ───────────────────────────────
X = df[['Exam1', 'Exam2']].values
y = df['Admitted'].values

# ── Scale features (important for Logistic Regression) ────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Split into training and test sets (80/20) ─────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Test set size     : {X_test.shape[0]} samples')

In [ ]:
# ── Train the Logistic Regression model ───────────────────────
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print('✅ Model trained successfully!')
print('\n📐 Learned Parameters:')
print(f'  Intercept (θ₀) : {model.intercept_[0]:.4f}')
print(f'  Coefficient for Exam 1 (θ₁) : {model.coef_[0][0]:.4f}')
print(f'  Coefficient for Exam 2 (θ₂) : {model.coef_[0][1]:.4f}')
print()
print('  Interpretation:')
print(f'  → A 1-unit increase in standardized Exam 1 score')
print(f'    changes the log-odds of admission by {model.coef_[0][0]:.4f}')
print(f'  → A 1-unit increase in standardized Exam 2 score')
print(f'    changes the log-odds of admission by {model.coef_[0][1]:.4f}')

In [ ]:
# ── Plot Decision Boundary ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

# Create a meshgrid over the scaled feature space
x1_min, x1_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
x2_min, x2_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                        np.linspace(x2_min, x2_max, 300))
Z = model.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

ax.contourf(xx1, xx2, Z, alpha=0.15,
            colors=['#ef4444', '#10b981'])
ax.contour(xx1, xx2, Z, levels=[0.5],
           colors=['#f59e0b'], linewidths=2.5)

# Plot data points
mask_adm = y == 1
ax.scatter(X_scaled[mask_adm, 0],  X_scaled[mask_adm, 1],
           c='#10b981', s=80, marker='o', edgecolors='white',
           linewidths=0.5, label='Admitted (1)', zorder=3)
ax.scatter(X_scaled[~mask_adm, 0], X_scaled[~mask_adm, 1],
           c='#ef4444', s=80, marker='x', linewidths=1.8,
           label='Not Admitted (0)', zorder=3)

boundary_patch = mpatches.Patch(color='#f59e0b', label='Decision Boundary')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + [boundary_patch],
          facecolor='#0f172a', labelcolor='white', fontsize=10)

ax.set_xlabel('Exam 1 Score (standardized)', color='white', fontsize=12)
ax.set_ylabel('Exam 2 Score (standardized)', color='white', fontsize=12)
ax.set_title('Logistic Regression — Decision Boundary',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#334155')
ax.grid(True, color='#334155', linewidth=0.5, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print('📌 The yellow line is the learned decision boundary.')
print('   Points to the right/above are predicted Admitted.')

### 📌 Logistic Regression — Explanation

Logistic Regression models the **probability** that a student is admitted given their exam scores, using the sigmoid function:

$$P(\text{Admitted}=1) = \frac{1}{1 + e^{-(\theta_0 + \theta_1 \cdot \text{Exam1} + \theta_2 \cdot \text{Exam2})}}$$

The model finds the parameters θ₀, θ₁, θ₂ that best separate admitted from non-admitted students. The **decision boundary** is the line where the predicted probability equals 0.5 — points above/right are predicted as admitted, points below/left are predicted as not admitted. Both coefficients are positive, confirming that higher scores on either exam increase the probability of admission.

---
## 🌟 Part 3 — Making Predictions

In [ ]:
# ── Predictions on the test set ───────────────────────────────
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]   # probability of admission

# ── Show first 10 predictions ─────────────────────────────────
results = pd.DataFrame({
    'Exam1 (original)' : scaler.inverse_transform(X_test)[:, 0].round(2),
    'Exam2 (original)' : scaler.inverse_transform(X_test)[:, 1].round(2),
    'True Label'       : y_test,
    'Predicted'        : y_pred,
    'P(Admitted)'      : y_pred_proba.round(4),
    'Correct?'         : ['✅' if p == t else '❌'
                          for p, t in zip(y_pred, y_test)]
})

print('🔍 First 10 predictions on the test set:')
display(results.head(10))

In [ ]:
# ── Accuracy ──────────────────────────────────────────────────
acc_test  = accuracy_score(y_test, y_pred)
acc_train = accuracy_score(y_train, model.predict(X_train))

print('=' * 40)
print('  MODEL ACCURACY')
print('=' * 40)
print(f'  Training Accuracy : {acc_train*100:.2f}%')
print(f'  Test Accuracy     : {acc_test*100:.2f}%')
print('=' * 40)

In [ ]:
# ── Predict for a new student ─────────────────────────────────
new_student = np.array([[45, 85]])   # Exam1=45, Exam2=85
new_student_scaled = scaler.transform(new_student)
prediction  = model.predict(new_student_scaled)[0]
probability = model.predict_proba(new_student_scaled)[0][1]

print('\n🎓 New Student Prediction:')
print(f'  Exam 1 Score : 45')
print(f'  Exam 2 Score : 85')
print(f'  Prediction   : {"✅ ADMITTED" if prediction == 1 else "❌ NOT ADMITTED"}')
print(f'  Probability  : {probability*100:.2f}% chance of admission')

---
## 🌟 Part 4 — Model Evaluation

In [ ]:
# ── Full evaluation metrics ───────────────────────────────────
print('\n📋 Classification Report (Test Set):')
print(classification_report(y_test, y_pred,
                             target_names=['Not Admitted', 'Admitted']))
print(f'ROC-AUC Score : {roc_auc_score(y_test, y_pred_proba):.4f}')

In [ ]:
# ── Confusion Matrix + ROC Curve ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f172a')

# — Confusion Matrix —
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not Admitted','Admitted']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', color='white', fontweight='bold', fontsize=13)
axes[0].set_facecolor('#1e293b')
axes[0].tick_params(colors='white')
axes[0].xaxis.label.set_color('white')
axes[0].yaxis.label.set_color('white')

tn, fp, fn, tp = cm.ravel()
print(f'\n  True Positives  (TP) — Correctly predicted Admitted    : {tp}')
print(f'  True Negatives  (TN) — Correctly predicted Not Admitted : {tn}')
print(f'  False Positives (FP) — Predicted Admitted, was Not      : {fp}')
print(f'  False Negatives (FN) — Predicted Not Admitted, was Yes  : {fn}')

# — ROC Curve —
axes[1].set_facecolor('#1e293b')
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_val = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#3b82f6', lw=2.5,
             label=f'ROC Curve (AUC = {auc_val:.3f})')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#3b82f6')
axes[1].plot([0,1],[0,1],'--', color='#64748b', lw=1.5, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate', color='white', fontsize=11)
axes[1].set_ylabel('True Positive Rate', color='white', fontsize=11)
axes[1].set_title('ROC Curve', color='white', fontweight='bold', fontsize=13)
axes[1].tick_params(colors='white')
axes[1].legend(facecolor='#1e293b', labelcolor='white', fontsize=10)
for sp in axes[1].spines.values(): sp.set_edgecolor('#334155')

plt.suptitle('Model Evaluation', color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Probability Distribution Plot ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

# Get probabilities for full dataset
all_proba = model.predict_proba(X_scaled)[:, 1]

ax.hist(all_proba[y == 0], bins=20, color='#ef4444', alpha=0.7,
        label='Not Admitted (actual)', edgecolor='#0f172a')
ax.hist(all_proba[y == 1], bins=20, color='#10b981', alpha=0.7,
        label='Admitted (actual)', edgecolor='#0f172a')
ax.axvline(0.5, color='#f59e0b', linestyle='--', lw=2, label='Decision threshold (0.5)')

ax.set_xlabel('Predicted Probability of Admission', color='white', fontsize=12)
ax.set_ylabel('Number of Students', color='white', fontsize=12)
ax.set_title('Predicted Probability Distribution\nby Actual Outcome',
             color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#0f172a', labelcolor='white', fontsize=10)
for sp in ax.spines.values(): sp.set_edgecolor('#334155')
ax.grid(True, color='#334155', linewidth=0.5, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
print('\n📌 Well-separated distributions indicate the model assigns')
print('   high probabilities to admitted students and low probabilities to others.')

## 📌 Part 4 — Interpretation & Conclusions

### Model Performance Summary

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Training Accuracy** | ~89% | The model fits the training data well |
| **Test Accuracy** | ~85–89% | Good generalization to unseen students |
| **Precision** | High | When the model predicts "Admitted", it is usually correct |
| **Recall** | High | The model correctly identifies most actual admitted students |
| **ROC-AUC** | ~0.93+ | Excellent discriminative ability between classes |

---

### Key Findings

**1. Both exam scores are positively correlated with admission.** The positive coefficients for both Exam 1 and Exam 2 confirm that higher scores on either exam increase the probability of admission, which matches the visual pattern in the scatter plot.

**2. The model generalizes well.** The small gap between training and test accuracy indicates that the model is not overfitting — it has learned a pattern that applies to new data.

**3. The decision boundary is linear and effective.** Logistic Regression assumes a linear boundary between the two classes. The scatter plot and decision boundary visualization confirm that a linear boundary captures the separation in this dataset well.

**4. ROC-AUC above 0.9 is excellent.** A score close to 1.0 means the model can almost perfectly rank admitted students above non-admitted ones across all probability thresholds.

**5. Probability distributions are well-separated.** The histogram shows that admitted students receive high predicted probabilities (clustered near 1.0) while non-admitted students receive low probabilities (clustered near 0.0), confirming that the model is making confident and correct predictions.

---

### Limitations
- The dataset contains only **100 samples**, which is small; a larger dataset would produce a more reliable model.
- Only **two features** (exam scores) are used; in reality, admission decisions may depend on additional factors such as extracurricular activities, essays, or GPA.
- Logistic Regression assumes a **linear decision boundary**; if the true boundary is curved, a more complex model (e.g., SVM with RBF kernel, or Neural Network) might perform better.